# MCI Criteria Verification and Extraction

This notebook identifies MCI visits in the longitudinal DELCODE metadata and exports all visits plus first-visit-per-patient outputs.

## Criteria Table

| Cohort | Cognitive Rule | Biomarker Rule | Decision |
|---|---|---|---|
| MCI - Criterion 1 | `26 <= mmstot <= 30` AND `0.5 <= cdrglobal <= 1` | Not required | Included in MCI |
| MCI - Criterion 2 | `mmstot > 26` AND `cdrglobal < 0.5` | `ratio_Abeta42_40 <= 0.08` AND `ratio_Abeta42_phosphotau181 < 9.68` | Included in MCI |
| MCI - Criterion 3 | `24 <= mmstot <= 28` AND `cdrglobal == 0.5` | `ratio_Abeta42_40 > 0.08` AND `ratio_Abeta42_phosphotau181 >= 9.68` | Included in MCI |
| Combined MCI | Any of Criterion 1, 2, or 3 | According to criterion matched | Included in MCI output |

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Centralized paths ──────────────────────────────────────────────────
BASE = Path('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION')
INTERMEDIATE = BASE / 'data' / 'intermediate'

input_csv = INTERMEDIATE / 'combined' / 'all_visits.csv'
output_dir = INTERMEDIATE / 'mci'
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'highlighted.xlsx'
output_all_csv = output_dir / 'all_visits.csv'
output_csv = output_dir / 'first_visit.csv'

# Load data
df = pd.read_csv(input_csv)

print(f"Input CSV: {input_csv}")
print(f"Output directory: {output_dir}")
print(f"Total rows: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head(3)


Input CSV: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/all_visits.csv
Output directory: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci
Total rows: 4605

Columns: ['file', 'Repseudonym', 'visdate', 'sex', 'age', 'prmdiag', 'mmstot', 'cdrtot', 'cdrglobal', 'Abeta38', 'Abeta40', 'Abeta42', 'totaltau', 'phosphotau181', 'ratio_Abeta42_40', 'ratio_Abeta42_phosphotau181', 'visnam', 'scan_date', 'scan_year']

First few rows:


,file,Repseudonym,visdate,sex,age,prmdiag,mmstot,cdrtot,cdrglobal,Abeta38,Abeta40,Abeta42,totaltau,phosphotau181,ratio_Abeta42_40,ratio_Abeta42_phosphotau181,visnam,scan_date,scan_year
0,BASELINE,011d501d1,01-10-2015,f,70.08,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08-10-2015,M0
1,FOLLOWUP,011d501d1,12-10-2016,f,71.11,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Follow-up 1,11-11-2016,M12
2,FOLLOWUP,011d501d1,20-09-2017,f,72.05,0,30,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Follow-up 2,NaN,NaN


In [10]:
# Check for missing values in key columns
key_cols = ['mmstot', 'cdrglobal', 'ratio_Abeta42_40', 'ratio_Abeta42_phosphotau181']
print("Missing values in key columns:")
for col in key_cols:
    missing = df[col].isna().sum()
    print(f"  {col}: {missing} ({100*missing/len(df):.1f}%)")

# Convert key columns to numeric
df['mmstot'] = pd.to_numeric(df['mmstot'], errors='coerce')
df['cdrglobal'] = pd.to_numeric(df['cdrglobal'], errors='coerce')
df['ratio_Abeta42_40'] = pd.to_numeric(df['ratio_Abeta42_40'], errors='coerce')
df['ratio_Abeta42_phosphotau181'] = pd.to_numeric(df['ratio_Abeta42_phosphotau181'], errors='coerce')

Missing values in key columns:
  mmstot: 99 (2.1%)
  cdrglobal: 67 (1.5%)
  ratio_Abeta42_40: 3829 (83.1%)
  ratio_Abeta42_phosphotau181: 3830 (83.2%)


In [11]:
# ── Criteria 1: MMSE 26-30 AND CDR 0.5-1 (biomarkers not important) ──
criteria_1 = (
    (df['mmstot'] >= 26) & (df['mmstot'] <= 30) &
    (df['cdrglobal'] >= 0.5) & (df['cdrglobal'] <= 1)
)

# ── Criteria 2: Cognitively normal AND abnormal biomarkers ──
# Cognitively normal: MMSE > 26 AND CDR < 0.5
# Abnormal biomarkers: ratio_Abeta42_40 <= 0.08 AND ratio_Abeta42_phosphotau181 < 9.68
criteria_2 = (
    (df['mmstot'] > 26) & (df['cdrglobal'] < 0.5) &
    (df['ratio_Abeta42_40'] <= 0.08) &
    (df['ratio_Abeta42_phosphotau181'] < 9.68)
)

# ── Criteria 3: MMSE 24-28 AND CDR = 0.5 AND biomarkers normal ──
# Biomarkers normal: ratio_Abeta42_40 > 0.08 AND ratio_Abeta42_phosphotau181 >= 9.68
criteria_3 = (
    (df['mmstot'] >= 24) & (df['mmstot'] <= 28) &
    (df['cdrglobal'] == 0.5) &
    (df['ratio_Abeta42_40'] > 0.08) &
    (df['ratio_Abeta42_phosphotau181'] >= 9.68)
)

# ── Combined MCI: any of the 3 criteria ──
meets_mci = criteria_1 | criteria_2 | criteria_3

# Store index sets per criteria
idx_criteria_1 = set(df[criteria_1].index)
idx_criteria_2 = set(df[criteria_2].index)
idx_criteria_3 = set(df[criteria_3].index)
idx_mci_all = set(df[meets_mci].index)

print(f"Rows meeting MCI Criteria 1 (MMSE 26-30, CDR 0.5-1): {len(idx_criteria_1)}")
print(f"Rows meeting MCI Criteria 2 (cognitive normal, abnormal biomarkers): {len(idx_criteria_2)}")
print(f"Rows meeting MCI Criteria 3 (MMSE 24-28, CDR=0.5, normal biomarkers): {len(idx_criteria_3)}")
print(f"\nRows meeting ANY MCI criteria (union): {len(idx_mci_all)}")

# Check overlaps
overlap_12 = idx_criteria_1 & idx_criteria_2
overlap_13 = idx_criteria_1 & idx_criteria_3
overlap_23 = idx_criteria_2 & idx_criteria_3
print(f"\nOverlaps:")
print(f"  Criteria 1 ∩ 2: {len(overlap_12)}")
print(f"  Criteria 1 ∩ 3: {len(overlap_13)}")
print(f"  Criteria 2 ∩ 3: {len(overlap_23)}")

df_mci = df[meets_mci].copy()
print(f"\n=== EXTRACTION RESULT ===")
print(f"Total MCI rows: {len(df_mci)}")
df_mci.to_csv(output_all_csv, index=False)
df_mci.head(3)

Rows meeting MCI Criteria 1 (MMSE 26-30, CDR 0.5-1): 1237
Rows meeting MCI Criteria 2 (cognitive normal, abnormal biomarkers): 61
Rows meeting MCI Criteria 3 (MMSE 24-28, CDR=0.5, normal biomarkers): 42

Rows meeting ANY MCI criteria (union): 1304

Overlaps:
  Criteria 1 ∩ 2: 0
  Criteria 1 ∩ 3: 36
  Criteria 2 ∩ 3: 0

=== EXTRACTION RESULT ===
Total MCI rows: 1304


,file,Repseudonym,visdate,sex,age,prmdiag,mmstot,cdrtot,cdrglobal,Abeta38,Abeta40,Abeta42,totaltau,phosphotau181,ratio_Abeta42_40,ratio_Abeta42_phosphotau181,visnam,scan_date,scan_year
8,BASELINE,01490270a,27-02-2017,f,83.83,5,26.0,1,0.5,3159.720249,7415.327129,293.68265,638.408068,84.587,0.039605,3.47196,NaN,01-03-2017,M0
14,FOLLOWUP,0165a12e0,21-09-2015,m,64.31,1,28.0,0.5,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Follow-up 1,24-09-2015,M12
27,BASELINE,0209d0b35,18-12-2017,m,72.30,1,30.0,1,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13-05-2018,M0


In [12]:
# Parse visdate before sorting to get correct chronological order
df_mci['_vd'] = pd.to_datetime(df_mci['visdate'], dayfirst=True, errors='coerce')

df_mci_first_visit = (df_mci
    .sort_values(['Repseudonym', '_vd'])
    .drop_duplicates(subset=['Repseudonym'], keep='first')
    .copy())

df_mci = df_mci.drop(columns=['_vd'])
df_mci_first_visit = df_mci_first_visit.drop(columns=['_vd'])

print(f'First MCI visit per patient: {len(df_mci_first_visit)} patients')
df_mci_first_visit[['Repseudonym', 'visdate', 'prmdiag', 'mmstot', 'cdrglobal']].head(3)


First MCI visit per patient: 490 patients


,Repseudonym,visdate,prmdiag,mmstot,cdrglobal
8,01490270a,27-02-2017,5,26.0,0.5
14,0165a12e0,21-09-2015,1,28.0,0.5
27,0209d0b35,18-12-2017,1,30.0,0.5


## Add first_mci_visit Column
Set `first_mci_visit` to the `scan_date` of the earliest MCI visit per patient.

In [13]:
# ── Add first_mci_visit column (scan_date with visdate fallback) ───────
# For each patient, first_mci_visit = scan_date of earliest MCI visit,
# falling back to visdate if scan_date is missing. Format: DD-MM-YYYY.

def get_first_visit_date(df):
    """Get earliest visit date per patient, preferring scan_date, falling back to visdate."""
    tmp = df.copy()
    tmp['_date'] = pd.to_datetime(tmp.get('scan_date', pd.Series(dtype=str)), dayfirst=True, errors='coerce')
    tmp['_date'] = tmp['_date'].fillna(pd.to_datetime(tmp['visdate'], dayfirst=True, errors='coerce'))
    return (tmp.dropna(subset=['_date'])
            .sort_values('_date')
            .drop_duplicates(subset=['Repseudonym'], keep='first')
            .set_index('Repseudonym')['_date'])

first_mci = get_first_visit_date(df_mci)

# Map back — all visits and first visit
df_mci['first_mci_visit'] = df_mci['Repseudonym'].map(first_mci).dt.strftime('%d-%m-%Y')
df_mci_first_visit['first_mci_visit'] = df_mci_first_visit['Repseudonym'].map(first_mci).dt.strftime('%d-%m-%Y')

print(f'first_mci_visit populated: {df_mci["first_mci_visit"].notna().sum()} / {len(df_mci)} rows')
print(f'Sample:')
df_mci[['Repseudonym', 'visdate', 'scan_date', 'first_mci_visit']].head(5)

# Re-save all-visits CSV with first_mci_visit column
df_mci.to_csv(output_all_csv, index=False)
print(f'\n✓ MCI all-visits CSV re-saved with first_mci_visit column')


first_mci_visit populated: 1304 / 1304 rows
Sample:

✓ MCI all-visits CSV re-saved with first_mci_visit column


In [14]:
# Save first visit per patient to CSV
df_mci_first_visit.to_csv(output_csv, index=False)

print(f"✓ MCI first visit CSV saved to: {output_csv}")
print(f"  Total patients: {len(df_mci_first_visit)}")
print(f"  File size: {output_csv.stat().st_size:,} bytes")

✓ MCI first visit CSV saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/first_visit.csv
  Total patients: 490
  File size: 64,372 bytes


In [15]:
# Summary
print("=" * 70)
print("MCI CRITERIA VERIFICATION SUMMARY")
print("=" * 70)
print(f"\nTotal rows in dataset: {len(df):,}")

print(f"\n>>> CRITERIA 1 (MMSE 26-30, CDR 0.5-1, biomarkers irrelevant): {len(idx_criteria_1)}")
if len(idx_criteria_1) > 0:
    c1 = df.loc[list(idx_criteria_1)]
    print(f"    mmstot range: {c1['mmstot'].min():.0f} - {c1['mmstot'].max():.0f}")
    print(f"    cdrglobal range: {c1['cdrglobal'].min():.2f} - {c1['cdrglobal'].max():.2f}")

print(f"\n>>> CRITERIA 2 (cognitive normal + abnormal biomarkers): {len(idx_criteria_2)}")
if len(idx_criteria_2) > 0:
    c2 = df.loc[list(idx_criteria_2)]
    print(f"    mmstot range: {c2['mmstot'].min():.0f} - {c2['mmstot'].max():.0f}")
    print(f"    cdrglobal range: {c2['cdrglobal'].min():.2f} - {c2['cdrglobal'].max():.2f}")
    print(f"    ratio_Abeta42_40 range: {c2['ratio_Abeta42_40'].min():.4f} - {c2['ratio_Abeta42_40'].max():.4f}")
    print(f"    ratio_Abeta42_phosphotau181 range: {c2['ratio_Abeta42_phosphotau181'].min():.2f} - {c2['ratio_Abeta42_phosphotau181'].max():.2f}")

print(f"\n>>> CRITERIA 3 (MMSE 24-28, CDR=0.5, normal biomarkers): {len(idx_criteria_3)}")
if len(idx_criteria_3) > 0:
    c3 = df.loc[list(idx_criteria_3)]
    print(f"    mmstot range: {c3['mmstot'].min():.0f} - {c3['mmstot'].max():.0f}")
    print(f"    cdrglobal range: {c3['cdrglobal'].min():.2f} - {c3['cdrglobal'].max():.2f}")
    print(f"    ratio_Abeta42_40 range: {c3['ratio_Abeta42_40'].min():.4f} - {c3['ratio_Abeta42_40'].max():.4f}")
    print(f"    ratio_Abeta42_phosphotau181 range: {c3['ratio_Abeta42_phosphotau181'].min():.2f} - {c3['ratio_Abeta42_phosphotau181'].max():.2f}")

print(f"\n>>> COMBINED MCI (union of all 3): {len(idx_mci_all)} rows, {len(df_mci_first_visit)} unique patients")
print(f"\nAll MCI rows get dark orange highlighting in Excel output.")

MCI CRITERIA VERIFICATION SUMMARY

Total rows in dataset: 4,605

>>> CRITERIA 1 (MMSE 26-30, CDR 0.5-1, biomarkers irrelevant): 1237
    mmstot range: 26 - 30
    cdrglobal range: 0.50 - 1.00

>>> CRITERIA 2 (cognitive normal + abnormal biomarkers): 61
    mmstot range: 27 - 30
    cdrglobal range: 0.00 - 0.00
    ratio_Abeta42_40 range: 0.0290 - 0.0719
    ratio_Abeta42_phosphotau181 range: 2.56 - 9.37

>>> CRITERIA 3 (MMSE 24-28, CDR=0.5, normal biomarkers): 42
    mmstot range: 24 - 28
    cdrglobal range: 0.50 - 0.50
    ratio_Abeta42_40 range: 0.0808 - 0.1304
    ratio_Abeta42_phosphotau181 range: 12.42 - 39.29

>>> COMBINED MCI (union of all 3): 1304 rows, 490 unique patients

All MCI rows get dark orange highlighting in Excel output.


## Highlight Rows Which Present MCI

In [16]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

display_cols = ['file', 'Repseudonym', 'visdate', 'scan_date', 'scan_year', 'prmdiag',
                'mmstot', 'cdrglobal', 'ratio_Abeta42_40',
                'ratio_Abeta42_phosphotau181']

# Build tag columns using pd.Series to avoid numpy.matrix compat bug
mci_criteria_col = pd.Series('', index=df.index, dtype=str)
for idx in idx_mci_all:
    tags = []
    if idx in idx_criteria_1:
        tags.append('C1')
    if idx in idx_criteria_2:
        tags.append('C2')
    if idx in idx_criteria_3:
        tags.append('C3')
    mci_criteria_col.at[idx] = ','.join(tags)

first_visit_col = pd.Series('', index=df.index, dtype=str)
idx_mci_first = set(df_mci_first_visit.index)
for idx in idx_mci_first:
    first_visit_col.at[idx] = 'YES'

df_export = df[display_cols].copy()
df_export = df_export.assign(
    mci_criteria=mci_criteria_col,
    first_visit=first_visit_col
)

# Write to Excel with highlighting
wb = Workbook()
ws = wb.active
ws.title = 'MCI Visits'

# Dark orange fill for MCI rows
orange_fill = PatternFill(start_color='FBE7CF', end_color='FBE7CF', fill_type='solid')
white_font = Font(color='FFFFFF', bold=False)
header_fill = PatternFill(start_color='333333', end_color='333333', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)

export_cols = display_cols + ['mci_criteria', 'first_visit']
for col_idx, col_name in enumerate(export_cols, 1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for excel_row, (df_idx, row_data) in enumerate(df_export.iterrows(), 2):
    is_mci = df_idx in idx_mci_all
    for col_idx, col_name in enumerate(export_cols, 1):
        value = row_data[col_name]
        if pd.isna(value):
            value = ''
        cell = ws.cell(row=excel_row, column=col_idx, value=value)
        if is_mci:
            cell.fill = orange_fill

# Auto-adjust column widths
for col_idx, col_name in enumerate(export_cols, 1):
    max_len = len(str(col_name))
    for row in ws.iter_rows(min_row=2, max_row=min(100, ws.max_row), min_col=col_idx, max_col=col_idx):
        for cell in row:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = min(max_len + 2, 40)

wb.save(output_file)

print(f"✓ MCI visits highlighted Excel saved to: {output_file}")
print(f"  Total rows written: {len(df_export)}")
print(f"  MCI-highlighted rows (orange): {len(idx_mci_all)}")
print(f"  First-visit MCI patients: {len(idx_mci_first)}")
print(f"  File size: {output_file.stat().st_size:,} bytes")

✓ MCI visits highlighted Excel saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/highlighted.xlsx
  Total rows written: 4605
  MCI-highlighted rows (orange): 1304
  First-visit MCI patients: 490
  File size: 252,180 bytes
